# Re-analysis of the main run
Re-scoring under stricter rules, paired first-turn vs loop comparison, token-normalized energy, calibration metrics, clustered bootstrap, ROUGE-1/2 and truncation sensitivity.

Runs on the three CSV files of the main run; a CPU runtime is sufficient. Outputs are written to `agentic_energy_results_v4/analysis_A/`. Add a Hugging Face token as the Colab secret `HF_TOKEN` to enable input-token counting; without it that step is skipped.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Run all analyses

In [ ]:
# Re-analysis of the main run: re-scoring under stricter rules, paired first-turn vs loop comparison,
# token-normalized energy, calibration metrics, clustered bootstrap, ROUGE-1/2 and truncation sensitivity.
# Outputs are written to agentic_energy_results_v4/analysis_A/ as CSV files.
import os, re, json, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets", "evaluate", "rouge_score", "scikit-learn"], check=False)
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import f1_score, roc_auc_score, brier_score_loss

DRIVE_PATH = "/content/drive" if os.path.ismount("/content/drive") else "/content/yeni_drive"
BASE = os.path.join(DRIVE_PATH, "MyDrive", "agentic_energy_results_v4")
OUT = os.path.join(BASE, "analysis_A"); os.makedirs(OUT, exist_ok=True)
SEED, N = 42, 200
CAT = ["small", "medium", "large"]
TASKS = ["classification", "math", "summarization"]
rng = np.random.default_rng(0)

# ---------------- 1. Data and reference answers ----------------
from datasets import load_dataset
def refs(task):
    if task == "classification":
        ds = load_dataset("fancyzhx/ag_news", split="test").shuffle(seed=SEED).select(range(N))
        names = ds.features["label"].names
        return [names[x] for x in ds["label"]]
    if task == "summarization":
        ds = load_dataset("abisee/cnn_dailymail", "3.0.0", split="test").shuffle(seed=SEED).select(range(N))
        return ds["highlights"]
    ds = load_dataset("openai/gsm8k", "main", split="test").shuffle(seed=SEED).select(range(N))
    return [re.search(r"####\s*([\-0-9,\.]+)", a).group(1).replace(",", "") for a in ds["answer"]]
REF = {t: refs(t) for t in TASKS}
print("References loaded:", {t: len(v) for t, v in REF.items()})

runs = {}
for tau in [5, 7, 9]:
    d = pd.read_csv(os.path.join(BASE, f"agentic_energy_v4_threshold{tau}.csv"))
    d["conf"] = d["confidence_history"].apply(json.loads)
    d["first_conf"] = d["conf"].str[0]; d["final_conf"] = d["conf"].str[-1]
    d["outs"] = d["per_turn_outputs"].apply(json.loads)
    d["ntok"] = d["n_new_tokens_history"].apply(json.loads)
    d["out_tokens_total"] = d["ntok"].apply(sum)
    d["first_out"] = d["outs"].str[0]
    d["reference"] = [REF[t][i] for t, i in zip(d["task"], d["sample_idx"])]
    runs[tau] = d
d7 = runs[7]

# ---------------- 2. Re-scoring ----------------
LABELS = ["world", "sports", "business", "sci/tech"]
def strict_cls(pred, ref):
    p = str(pred).lower()
    found = [l for l in LABELS if l in p]
    return int(len(found) == 1 and found[0] == ref.lower())
def lenient_cls(pred, ref):
    return int(ref.lower() in str(pred).lower())
def math_answer_line(pred, ref):
    p = str(pred)
    m = re.findall(r"[Aa]nswer\s*:\s*\$?\s*([\-]?\d[\d,]*\.?\d*)", p)
    nums = m if m else re.findall(r"[\-]?\d[\d,]*\.?\d*", p)
    if not nums: return 0, 1
    try: return int(abs(float(nums[-1].replace(",", "")) - float(ref)) < 1e-4), 0
    except ValueError: return 0, 1
def math_last_number(pred, ref):
    nums = re.findall(r"[\-]?\d[\d,]*\.?\d*", str(pred))
    try: return int(abs(float(nums[-1].replace(",", "")) - float(ref)) < 1e-4) if nums else 0
    except ValueError: return 0

import evaluate
rouge = evaluate.load("rouge")
def rouge_all(preds, refs_):
    return rouge.compute(predictions=preds, references=refs_, use_aggregator=False)

for tau, d in runs.items():
    c = d.task == "classification"; m = d.task == "math"; s = d.task == "summarization"
    d.loc[c, "final_strict"] = [strict_cls(p, r) for p, r in zip(d.loc[c, "final_output"], d.loc[c, "reference"])]
    d.loc[c, "first_strict"] = [strict_cls(p, r) for p, r in zip(d.loc[c, "first_out"], d.loc[c, "reference"])]
    d.loc[c, "first_lenient"] = [lenient_cls(p, r) for p, r in zip(d.loc[c, "first_out"], d.loc[c, "reference"])]
    fa = [math_answer_line(p, r) for p, r in zip(d.loc[m, "final_output"], d.loc[m, "reference"])]
    fi = [math_answer_line(p, r) for p, r in zip(d.loc[m, "first_out"], d.loc[m, "reference"])]
    d.loc[m, "final_strict"] = [x[0] for x in fa]; d.loc[m, "final_parsefail"] = [x[1] for x in fa]
    d.loc[m, "first_strict"] = [x[0] for x in fi]
    d.loc[m, "first_lenient"] = [math_last_number(p, r) for p, r in zip(d.loc[m, "first_out"], d.loc[m, "reference"])]
    rf = rouge_all(d.loc[s, "final_output"].astype(str).tolist(), d.loc[s, "reference"].tolist())
    r1 = rouge_all(d.loc[s, "first_out"].astype(str).tolist(), d.loc[s, "reference"].tolist())
    for k in ["rouge1", "rouge2", "rougeL"]:
        d.loc[s, f"final_{k}"] = rf[k]; d.loc[s, f"first_{k}"] = r1[k]
    d.loc[s, "final_strict"] = rf["rougeL"]; d.loc[s, "first_strict"] = r1["rougeL"]; d.loc[s, "first_lenient"] = r1["rougeL"]
    d["final_lenient"] = d["metric"] if "metric" in d else np.where(s, d["rougeL"], d["correct"])

# Lenient vs strict scoring rule (tau = 7)
sc = d7.groupby(["task", "model"]).agg(lenient=("final_lenient", "mean"), strict=("final_strict", "mean")).round(3)
sc["diff"] = (sc["strict"] - sc["lenient"]).round(3)
sc.to_csv(os.path.join(OUT, "scoring_rule_comparison.csv"))
print("\n[2] Lenient vs strict scoring (tau=7):\n", sc)
mf = d7[d7.task == "classification"].copy()
def pred_label(p):
    f = [l for l in LABELS if l in str(p).lower()]; return f[0] if len(f) == 1 else "invalid"
mf["pred"] = mf["final_output"].apply(pred_label); mf["ref"] = mf["reference"].str.lower()
macro = mf.groupby("model").apply(lambda g: f1_score(g["ref"], g["pred"], labels=LABELS, average="macro", zero_division=0)).round(3)
macro.to_csv(os.path.join(OUT, "classification_macro_f1.csv")); print("\nClassification macro-F1 (strict):\n", macro)
print("Math answer-line parse failures (tau=7):", int(d7.loc[d7.task == "math", "final_parsefail"].sum()))

# ---------------- 3. Paired comparison: first turn vs loop, all 200 samples per cell ----------------
def mcnemar(b, c):
    n = b + c
    return 1.0 if n == 0 else min(1.0, 2 * stats.binom.cdf(min(b, c), n, 0.5))
rows = []
for (task, model, cat), g in d7.groupby(["task", "model", "model_category"]):
    first, final = g["first_strict"].values, g["final_strict"].values
    if task == "summarization":
        diff = final - first
        boots = [rng.choice(diff, len(diff)).mean() for _ in range(2000)]
        rows.append(dict(task=task, model=model, category=cat, first=first.mean(), final=final.mean(),
                         gain=diff.mean(), ci_lo=np.percentile(boots, 2.5), ci_hi=np.percentile(boots, 97.5),
                         p=np.nan, n_changed=int((g["n_turns"] > 1).sum())))
    else:
        b = int(((first == 1) & (final == 0)).sum()); c = int(((first == 0) & (final == 1)).sum())
        diff = final - first
        boots = [rng.choice(diff, len(diff)).mean() for _ in range(2000)]
        rows.append(dict(task=task, model=model, category=cat, first=first.mean(), final=final.mean(),
                         gain=diff.mean(), ci_lo=np.percentile(boots, 2.5), ci_hi=np.percentile(boots, 97.5),
                         p=mcnemar(b, c), n_changed=int((g["n_turns"] > 1).sum()), worse=b, better=c))
paired = pd.DataFrame(rows).round(4); paired.to_csv(os.path.join(OUT, "paired_first_vs_loop.csv"), index=False)
print("\n[3] Paired first-turn vs loop (tau=7, strict scoring, n=200 each):\n", paired.to_string(index=False))

# ---------------- 4. Token-normalized energy ----------------
tok = (d7.groupby(["task", "model", "model_category"])
       .agg(energy=("energy_kwh", "mean"), out_tokens=("out_tokens_total", "mean"), elapsed=("elapsed_sec", "mean"),
            turns=("n_turns", "mean")).reset_index())
tok["J_per_out_token"] = tok["energy"] * 3.6e6 / tok["out_tokens"]
tok["tokens_per_s"] = tok["out_tokens"] / tok["elapsed"]
tok["avg_power_W"] = tok["energy"] * 3.6e6 / tok["elapsed"]
# Input token counts (optional; needs a Hugging Face token for the gated tokenizers)
try:
    from google.colab import userdata
    from huggingface_hub import login; login(userdata.get("HF_TOKEN"))
    from transformers import AutoTokenizer
    HF = {"llama-3.2-3b": "meta-llama/Llama-3.2-3B-Instruct", "qwen2.5-3b": "Qwen/Qwen2.5-3B-Instruct",
          "gemma-2-2b": "google/gemma-2-2b-it", "llama-3-8b": "meta-llama/Meta-Llama-3-8B-Instruct",
          "qwen2.5-7b": "Qwen/Qwen2.5-7B-Instruct", "mistral-7b": "mistralai/Mistral-7B-Instruct-v0.3",
          "llama-3-70b": "unsloth/llama-3-70b-Instruct-bnb-4bit", "qwen2.5-72b": "unsloth/Qwen2.5-72B-Instruct-bnb-4bit",
          "mistral-large": "unsloth/Mistral-Large-Instruct-2407-bnb-4bit"}
    def base_prompt(task, inp):
        if task == "summarization": return "Summarize the following news article in 2-3 sentences. Write the summary in English.\n\n" + inp[:3000]
        if task == "classification": return ("Classify the following news text into exactly ONE of these four categories: World, Sports, Business, Sci/Tech.\n\n"
            "Respond with ONLY the category name (World, Sports, Business or Sci/Tech). Do not add any explanation or other words.\n\n" f"Text: {inp[:2000]}\n\nCategory:")
        return "Solve the following math problem step by step. On the last line, write your final answer in the format 'Answer: <number>'.\n\n" f"Problem: {inp}"
    INP = {}
    INP["classification"] = load_dataset("fancyzhx/ag_news", split="test").shuffle(seed=SEED).select(range(N))["text"]
    INP["summarization"] = load_dataset("abisee/cnn_dailymail", "3.0.0", split="test").shuffle(seed=SEED).select(range(N))["article"]
    INP["math"] = load_dataset("openai/gsm8k", "main", split="test").shuffle(seed=SEED).select(range(N))["question"]
    in_rows = []
    for model, hid in HF.items():
        tk = AutoTokenizer.from_pretrained(hid)
        for task in TASKS:
            lens = [len(tk(tk.apply_chat_template([{"role": "user", "content": base_prompt(task, x)}], tokenize=False, add_generation_prompt=True), add_special_tokens=False)["input_ids"]) for x in INP[task]]
            in_rows.append(dict(task=task, model=model, in_tokens_first_turn=np.mean(lens)))
    tok = tok.merge(pd.DataFrame(in_rows), on=["task", "model"], how="left")
    print("\n[4] Input tokens counted with each model's tokenizer.")
except Exception as ex:
    print("\n[4] Input-token count skipped:", ex)
tok = tok.round(5); tok.to_csv(os.path.join(OUT, "token_normalized_energy.csv"), index=False)
print(tok.to_string(index=False))
# Parameter count vs output length: log(energy) ~ a*log(params) + b*log(out_tokens) + task effects
PARAMS = {"llama-3.2-3b": 3.2, "qwen2.5-3b": 3.1, "gemma-2-2b": 2.6, "llama-3-8b": 8.0, "qwen2.5-7b": 7.6,
          "mistral-7b": 7.2, "llama-3-70b": 70.6, "qwen2.5-72b": 72.7, "mistral-large": 123.0}
tok["log_params"] = np.log(tok["model"].map(PARAMS)); tok["log_tok"] = np.log(tok["out_tokens"]); tok["log_e"] = np.log(tok["energy"])
X = np.column_stack([np.ones(len(tok)), tok["log_params"], tok["log_tok"], pd.get_dummies(tok["task"], drop_first=True).values.astype(float)])
beta, *_ = np.linalg.lstsq(X, tok["log_e"], rcond=None)
print(f"\nlog(energy) ~ {beta[1]:.2f}*log(params) + {beta[2]:.2f}*log(out_tokens) + task effects  "
      f"(elasticities: params {beta[1]:.2f}, output tokens {beta[2]:.2f})")

# ---------------- 5. Calibration metrics ----------------
def ece(conf, correct, bins=10):
    p = np.asarray(conf) / 10.0; y = np.asarray(correct); e = 0.0
    for lo in np.arange(0, 1, 1 / bins):
        m = (p > lo) & (p <= lo + 1 / bins)
        if m.any(): e += m.mean() * abs(y[m].mean() - p[m].mean())
    return e
cal = []
s = d7[d7.task.isin(["classification", "math"])]
for scope, g in [("pooled", s)] + list(s.groupby("model")):
    y, c = g["final_strict"].values.astype(int), g["first_conf"].values
    cal.append(dict(scope=scope, n=len(g), mean_conf=c.mean() / 10, accuracy=y.mean(),
                    r_pb=stats.pointbiserialr(y, c)[0] if len(set(c)) > 1 else np.nan,
                    auroc=roc_auc_score(y, c) if 0 < y.mean() < 1 else np.nan,
                    brier=brier_score_loss(y, c / 10), ece=ece(c, y)))
cal = pd.DataFrame(cal).round(3); cal.to_csv(os.path.join(OUT, "calibration_metrics.csv"), index=False)
print("\n[5] Calibration (first-turn score/10 as probability; strict correctness):\n", cal.to_string(index=False))
rel = s.groupby("first_conf").agg(n=("final_strict", "size"), acc=("final_strict", "mean")).reset_index()
rel["conf_p"] = rel["first_conf"] / 10; rel.round(3).to_csv(os.path.join(OUT, "reliability_table.csv"), index=False)

# ---------------- 6. Clustered bootstrap (models resampled within category) ----------------
mm = d7.groupby(["task", "model", "model_category"])["energy_kwh"].mean().reset_index()
boot = []
for task in TASKS + ["all"]:
    sub = mm if task == "all" else mm[mm.task == task]
    ratios = []
    for _ in range(5000):
        samp = pd.concat([sub[sub.model_category == c].sample(frac=1, replace=True, random_state=int(rng.integers(1e9))) for c in CAT])
        ratios.append(samp[samp.model_category == "large"].energy_kwh.mean() / samp[samp.model_category != "large"].energy_kwh.mean())
    boot.append(dict(task=task, ratio_large_vs_rest=np.mean(ratios), ci_lo=np.percentile(ratios, 2.5), ci_hi=np.percentile(ratios, 97.5)))
boot = pd.DataFrame(boot).round(2); boot.to_csv(os.path.join(OUT, "clustered_bootstrap_energy_ratio.csv"), index=False)
print("\n[6] Clustered bootstrap (models resampled within category), large/rest energy ratio:\n", boot.to_string(index=False))

# ---------------- 7. Summarization: ROUGE-1/2/L and truncation sensitivity ----------------
ss = d7[d7.task == "summarization"]
rg = ss.groupby("model").agg(rouge1=("final_rouge1", "mean"), rouge2=("final_rouge2", "mean"), rougeL=("final_rougeL", "mean"),
                             truncated=("truncated_any", "mean")).round(4)
tr = ss.groupby(["model", "truncated_any"])["final_rougeL"].agg(["mean", "size"]).unstack("truncated_any").round(4)
rg.to_csv(os.path.join(OUT, "summarization_rouge_all.csv")); tr.to_csv(os.path.join(OUT, "summarization_truncation_sensitivity.csv"))
print("\n[7] Summarization ROUGE-1/2/L and truncation share:\n", rg, "\nROUGE-L by truncation (0 = not truncated, 1 = truncated):\n", tr)

# ---------------- 8. Measurement stability: per-turn energy across threshold runs ----------------
A = pd.concat(runs.values(), ignore_index=True); A["ept"] = A["energy_kwh"] / A["n_turns"]
ept = A.groupby(["task", "model", "threshold"])["ept"].mean().unstack("threshold")
stab = pd.DataFrame({"ratio_5_7": ept[5] / ept[7], "ratio_9_7": ept[9] / ept[7]}).round(3)
stab.to_csv(os.path.join(OUT, "measurement_stability_per_turn_energy.csv"))
print("\n[8] Per-turn energy stability across threshold runs (median ratios):", stab.median().round(3).to_dict())

# ---------------- 9. Energy per correct output ----------------
epc = d7.groupby(["task", "model", "model_category"]).agg(energy=("energy_kwh", "sum"), correct=("final_strict", "sum"), n=("final_strict", "size")).reset_index()
epc["kwh_per_correct"] = epc["energy"] / epc["correct"].replace(0, np.nan)
epc.round(7).to_csv(os.path.join(OUT, "energy_per_correct.csv"), index=False)
print("\n[9] kWh per correct output (classification, math) / per ROUGE-L unit (summarization):\n",
      epc.pivot(index="model", columns="task", values="kwh_per_correct").round(6))
print("\nDone. Outputs in", OUT)
